I. Script d'automatisation et pré-modification du fichier csv : Carburant_prix_fr > Dataset> Data.gouv 

In [1]:
pip install scipy sqlalchemy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: C:\Users\mehal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import scipy.stats as st
import os
from sqlalchemy import create_engine



In [3]:
pip install pymysql


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: C:\Users\mehal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:


# === 1. Télécharger automatiquement le fichier depuis data.gouv ===
url = "https://www.data.gouv.fr/api/1/datasets/r/edd67f5b-46d0-4663-9de9-e5db1c880160"
carburant_prix = pd.read_csv(url, sep=';', low_memory=False)

# === 2. Supprimer les colonnes inutiles ===
colonnes_a_supprimer_1 = [
    'services', 'prix', 'rupture', 'horaires détaillés', 'horaires',
    'Début rupture e85 (si temporaire)', 'Type rupture e85'
]
carburant_prix = carburant_prix.drop(columns=colonnes_a_supprimer_1, errors='ignore')

colonnes_a_supprimer_2 = [
    'Prix GPLc mis à jour le', 'Prix GPLc', 'Début rupture GPLc (si temporaire)',
    'Type rupture GPLc', 'Début rupture e10 (si temporaire)', 'Type rupture e10',
    'Début rupture sp98 (si temporaire)', 'Type rupture sp98',
    'Début rupture gazole (si temporaire)'
]
carburant_prix = carburant_prix.drop(columns=colonnes_a_supprimer_2, errors='ignore')

# === 3. Extraire latitude/longitude depuis la colonne 'geom' ===
carburant_prix[['latitude', 'longitude']] = carburant_prix['geom'].str.split(',', expand=True)
carburant_prix = carburant_prix.drop(columns=['geom'])

# === 4. Nettoyer les noms de colonnes ===
carburant_prix.columns = carburant_prix.columns.str.strip().str.lower().str.replace(' ', '_')

# === 5. Renommer les colonnes ===
carburant_prix = carburant_prix.rename(columns={
    'prix_gazole_mis_à_jour_le': 'prix_gazole_maj',
    'prix_sp95_mis_à_jour_le': 'prix_sp95_maj',
    'prix_e85_mis_à_jour_le': 'prix_e85_maj',
    'prix_e10_mis_à_jour_le': 'prix_e10_maj',
    'prix_sp98_mis_à_jour_le': 'prix_sp98_maj',
    'début_rupture_sp95_(si_temporaire)': 'debut_rupture_sp95',
    'automate_24-24_(oui/non)': 'automate_24_24',
    'services_proposés': 'service_propose',
    'début_rupture_sp95': 'debut_rupture_sp95',
    'département': 'departement'
})

# === 6. Conversion des dates ===
colonnes_dates = [
    "prix_gazole_maj", "prix_sp95_maj", "prix_e85_maj",
    "prix_e10_maj", "prix_sp98_maj", "debut_rupture_sp95"
]

for col in colonnes_dates:
    carburant_prix[col] = pd.to_datetime(carburant_prix[col], errors='coerce').dt.tz_localize(None)

# === 7. Sauvegarde finale ===

# 1. Sauvegarde sur le Bureau (pour toi)
chemin_bureau = os.path.join(os.path.expanduser("~"), "Desktop", "carburant_prix_nettoye.csv")
carburant_prix.to_csv(chemin_bureau, index=False, sep=';', encoding='utf-8-sig')

# 2. Sauvegarde dans Google Drive (pour archive)
chemin_drive = r"G:\Mon Drive\Carburant prix\carburant_prix_nettoye.csv"
carburant_prix.to_csv(chemin_drive, index=False, sep=';', encoding='utf-8-sig')

# 3. SAUVEGARDE POUR GITHUB (C'est celle-ci qui met à jour l'appli !)
chemin_github = r"C:\Users\mehal\Documents\Projets_Python\Carbunet\carburant_prix_nettoye.csv"
carburant_prix.to_csv(chemin_github, index=False, sep=';', encoding='utf-8-sig')

print(f"✅ Fichier mis à jour partout !")
print(f"📁 Bureau : {chemin_bureau}")
print(f"📁 GitHub : {chemin_github}")


from sqlalchemy import create_engine

# === 8. Connexion à ta base MariaDB ===
# 💡 Modifie les infos ci-dessous avec ta configuration réelle
utilisateur = "root"  # ou ton user MariaDB
mot_de_passe = "1212.Y8m"
hote = "localhost"
port = "3306"
base = "carburant_prix"  # nom de ta base dans DBeaver

# Création de la connexion SQLAlchemy
engine = create_engine(f"mysql+pymysql://{utilisateur}:{mot_de_passe}@{hote}:{port}/{base}?charset=utf8mb4")


# === 9. Envoi dans la table 'carburant_prix' ===
# Tu peux changer le nom de la table ici si tu veux
carburant_prix.to_sql(name='carburant_prix', con=engine, if_exists='replace', index=False)

print("✅ Données envoyées dans la base MariaDB (table : carburant_prix)")


✅ Fichier mis à jour partout !
📁 Bureau : C:\Users\mehal\Desktop\carburant_prix_nettoye.csv
📁 GitHub : C:\Users\mehal\Documents\Projets_Python\Carbunet\carburant_prix_nettoye.csv
✅ Données envoyées dans la base MariaDB (table : carburant_prix)


In [5]:
print(carburant_prix[['prix_gazole', 'prix_sp95', 'prix_sp98', 'prix_e10', 'prix_e85']].head())
print(carburant_prix[['prix_gazole_maj', 'prix_sp95_maj', 'prix_sp98_maj']].head())


   prix_gazole  prix_sp95  prix_sp98  prix_e10  prix_e85
0        2.349      2.099        NaN     2.049       NaN
1        2.469        NaN      2.099     2.019     0.729
2        2.259        NaN      2.129     1.999       NaN
3        2.489      2.109      2.059       NaN       NaN
4        2.449      2.109      2.169       NaN       NaN
      prix_gazole_maj       prix_sp95_maj       prix_sp98_maj
0 2026-04-07 09:53:45 2026-04-07 09:53:45                 NaT
1 2026-04-07 01:00:00                 NaT 2026-03-31 01:00:00
2 2026-04-03 08:08:45                 NaT 2026-04-03 08:08:45
3 2026-04-07 08:00:17 2026-04-07 08:00:17 2026-03-26 08:15:41
4 2026-03-31 18:01:54 2026-03-31 18:01:54 2026-03-31 18:01:54


In [6]:
carburant_prix.head()


,id,latitude,longitude,code_postal,pop,adresse,ville,prix_gazole_maj,prix_gazole,prix_sp95_maj,...,carburants_disponibles,carburants_indisponibles,carburants_en_rupture_temporaire,carburants_en_rupture_definitive,automate_24_24,service_propose,departement,code_departement,région,code_region
0,31620007,43.765374855500006,1.3699738693999999,31620,R,"1, rue de l'Ourméde",Castelnau-d'Estrétefonds,2026-04-07 09:53:45,2.349,2026-04-07 09:53:45,...,"Gazole,SP95,E10","E85,GPLc,SP98",NaN,E85;GPLc;SP98,Non,"Toilettes publiques,Boutique alimentaire,Resta...",Haute-Garonne,31,Occitanie,76.0
1,76270003,49.725277201557,1.4597834195908,76270,R,CENTRE COMMERCIAL SAINT JEAN,Neuville-Ferrières,2026-04-07 01:00:00,2.469,NaT,...,"Gazole,E85,E10,SP98","SP95,GPLc",NaN,SP95;GPLc,Oui,"Lavage automatique,Automate CB 24/24",Seine-Maritime,76,Normandie,28.0
2,49410002,47.359,-1.016,49410,R,zi de la chevallerie,Mauges-sur-Loire,2026-04-03 08:08:45,2.259,NaT,...,"Gazole,E10,SP98","SP95,E85,GPLc",NaN,SP95;E85;GPLc,Non,"Laverie,Station de gonflage,Piste poids lourds...",Maine-et-Loire,49,Pays de la Loire,52.0
3,3380002,46.372,2.468,3380,R,Rue Calaubys,Huriel,2026-04-07 08:00:17,2.489,2026-04-07 08:00:17,...,"Gazole,SP95,SP98","E85,GPLc,E10",NaN,E85;GPLc;E10,Oui,"Laverie,Boutique alimentaire,Boutique non alim...",Allier,03,Auvergne-Rhône-Alpes,84.0
4,24330005,45.123,0.86,24330,R,les guichoux,Saint-Pierre-de-Chignac,2026-03-31 18:01:54,2.449,2026-03-31 18:01:54,...,"Gazole,SP95,SP98","E85,GPLc,E10",NaN,E85;GPLc;E10,Oui,"Lavage manuel,Automate CB 24/24",Dordogne,24,Nouvelle-Aquitaine,75.0


In [7]:
mask = carburant_prix['adresse'].str.contains("Gier", case=False, na=False) | \
       carburant_prix['adresse'].str.contains("Giers", case=False, na=False)

carburant_prix[mask][['adresse','ville', 'prix_gazole', 'prix_gazole_maj']]



,adresse,ville,prix_gazole,prix_gazole_maj
5692,24 BOULEVARD AMBROISE BRUGIERE,Clermont-Ferrand,NaN,NaT
5912,Lieu-dit Les Grangiers,Luc-en-Diois,2.440,2026-04-03 15:01:39
6051,AUTOROUTE A47 - AIRE DU PAYS DU GIERS,Saint-Chamond,NaN,NaT
6358,AUTOROUTE A47 - AIRE DE SAINT ROMAIN EN GIER,Saint-Romain-en-Gier,2.619,2026-04-07 06:19:21
7311,25 RUE LOUIS AMARGIER,Saugues,2.299,2026-04-03 15:50:10
8305,Rue du Plat du Gier,L'Horme,2.459,2026-04-07 07:10:51
